In [1]:
import requests
import json
import scanpy as sc
import os
import pandas as pd

from perturbgen.configs import ROOT

In [2]:
# set wd
os.chdir(ROOT)
print('Current working directory:', ROOT)

Current working directory: /lustre/scratch126/cellgen/lotfollahi/kl11


In [3]:
adata_clustered = sc.read_h5ad('T_perturb/res/hspc/perturbation_5k_res/summary_plots/20250723_leiden_pert_cls_annotated.h5ad')

In [4]:
res_dir = '/lustre/scratch126/cellgen/lotfollahi/kl11/T_perturb/res/hspc/OT'
# create directory if it does not exist
if not os.path.exists(res_dir):
    os.makedirs(res_dir)

In [5]:
ot_df = pd.read_csv(f'{res_dir}/20251114_OT.csv')

In [6]:
relevant_therapeutic_areas = [
    'measurement',
    'cardiovascular disease',
    'genetic, familial or congenital disease',
    'hematologic disease',
    'immune system disease',
    'phenotype',
    'infectious disease',
    'cancer or benign tumor',
    'nutritional or metabolic disease',
    'developmental defect during embryogenesis'
]

In [7]:
ot_df = ot_df[ot_df.therapeutic_area_name.isin(relevant_therapeutic_areas)]

In [8]:
# filter out columns where chembl eva gene_burden genomics_england are all Nan and gwas_credible_sets < 0.5
# select disease_name based on the filtered_df
ot_df_filtered = ot_df[
    (ot_df['gwas_credible_sets'] >= 0.5 # filter locus_to_gene score
    )  | 
    (
        ot_df['eva'] >= 0.5  # 0.5 at least risk factor https://platform-docs.opentargets.org/evidence#clinvar
    ) |
    (
        ot_df['gene_burden'] >= 0
    ) |
    (
        ot_df['genomics_england'] >= 0
    )
]

In [9]:
ot_df_filtered[ot_df_filtered.therapeutic_area_name == 'measurement'].disease_name.unique().tolist()[:500]

['total blood protein measurement',
 'low density lipoprotein cholesterol measurement',
 'sex hormone-binding globulin measurement',
 'aspartate aminotransferase measurement',
 'protein measurement',
 'atrial natriuretic factor measurement',
 'complement C3 measurement',
 'monocyte count',
 'myeloperoxidase measurement',
 'drug use measurement',
 'BMI-adjusted waist-hip ratio',
 'BMI-adjusted waist circumference',
 'complement factor H measurement',
 'complement factor H-related protein 5 measurement',
 'interferon gamma measurement',
 'interleukin 8 measurement',
 'properdin measurement',
 'TNF-related apoptosis-inducing ligand measurement',
 'prothrombin time measurement',
 'osteocalcin measurement',
 'tissue factor measurement',
 'protein binding measurement',
 'C-C motif chemokine 20 measurement',
 'aurora kinase A measurement',
 'C-C motif chemokine 27 measurement',
 'carbonic anhydrase 7 measurement',
 'cytokine receptor-like factor 2 measurement',
 'diablo homolog, mitochondrial

In [10]:
# filter for disease_name in measurement with at least 5 associated genes
# -> GWAS traits with at least 5 genes
measurement_diseases = ot_df_filtered[ot_df_filtered.therapeutic_area_name == 'measurement']
filter_ = measurement_diseases.disease_name.value_counts()[measurement_diseases.disease_name.value_counts() >= 5].index
measurement_diseases = measurement_diseases[measurement_diseases.disease_name.isin(filter_)]

In [11]:
lineage_map = {
    # --- Erythroid lineage ---
    "erythrocyte count": "erythroid",
    "erythrocyte volume": "erythroid",
    "erythrocyte attribute": "erythroid",
    "hemoglobin measurement": "erythroid",
    "hemoglobin a1 measurement": "erythroid",
    "hemoglobin a1c measurement": "erythroid",  # if present
    "mean corpuscular hemoglobin": "erythroid",
    "mean corpuscular hemoglobin concentration": "erythroid",
    "hematocrit": "erythroid",
    "red cell distribution width": "erythroid",
    "reticulocyte count": "erythroid",
    "reticulocyte amount": "erythroid",
    "mean reticulocyte volume": "erythroid",
    "red blood cell density": "erythroid",
    "serum iron amount": "erythroid",

    # --- Megakaryocytic (platelet) lineage ---
    "platelet count": "megakaryocytic",
    "platelet volume": "megakaryocytic",
    "platelet crit": "megakaryocytic",
    "platelet component distribution width": "megakaryocytic",
    "immature platelet count": "megakaryocytic",
    "immature platelet measurement": "megakaryocytic",
    # "platelet-to-lymphocyte ratio": "megakaryocytic + lymphoid (mixed index)",

    # --- 'BaEoMa', lineage ---
    "eosinophil count": "basophilic/eosinophilic",
    "basophil count": "basophilic/eosinophilic",
    "eosinophil measurement": "basophilic/eosinophilic",
    "basophil measurement": "basophilic/eosinophilic",

    # --- 'Myeloid disorders', lineage ---
    "neutrophil count": "myeloid (granulocytic)",
    "granulocyte count": "myeloid (granulocytic)",
    # "granulocyte percentage of myeloid white cells": "myeloid (granulocytic)",
    "neutrophil measurement": "myeloid (granulocytic)",

    "myeloid leukocyte count": "myeloid",
    # "sum of basophil and neutrophil counts": "myeloid (granulocytic)",
    # "sum of neutrophil and eosinophil counts": "myeloid (granulocytic)",
    # "neutrophil percentage of leukocytes": "myeloid (granulocytic)",
    # "neutrophil percentage of granulocytes": "myeloid (granulocytic)",
    # "eosinophil percentage of leukocytes": "myeloid (granulocytic)",
    # "eosinophil percentage of granulocytes": "myeloid (granulocytic)",
    # "basophil percentage of leukocytes": "myeloid (granulocytic)",
    # monocytes
    "monocyte count": "myeloid (monocytic)",
    "monocyte measurement": "myeloid (monocytic)",
    "monocyte percentage of leukocytes": "myeloid (monocytic)",
    # combined ratios
    # "neutrophil-to-lymphocyte ratio": "myeloid + lymphoid (mixed index)",
    "lymphocyte:monocyte ratio": "lymphoid + myeloid (mixed index)",

    # --- Lymphoid lineage ---
    "lymphocyte count": "lymphoid",
    "lymphocyte amount": "lymphoid",
    "lymphocyte percentage of leukocytes": "lymphoid",
    # global white cell traits
    "leukocyte quantity": "pan-leukocytic",
    "hematological measurement": "pan-hematologic",
}
# first filter ot_df to only those in lineage_map
measurement_df = ot_df[ot_df.disease_name.isin(lineage_map.keys())].copy()
measurement_df['lineage_category'] = measurement_df['disease_name'].map(lineage_map)

In [12]:
ot_df_filtered[ot_df_filtered.therapeutic_area_name != 'measurement'].disease_name.unique().tolist()[:100]

['age-related macular degeneration',
 'IGA glomerulonephritis',
 'wet macular degeneration',
 'drug allergy',
 'atrophic macular degeneration',
 'Visual impairment',
 'Retinal dystrophy',
 'Respiratory insufficiency',
 'optic choroid disorder',
 'complement factor H deficiency',
 'age related macular degeneration 4',
 'atypical hemolytic-uremic syndrome with I factor anomaly',
 'atypical hemolytic-uremic syndrome',
 'atypical hemolytic-uremic syndrome with H factor anomaly',
 'COVID-19',
 'dry age related macular degeneration',
 'Familial drusen',
 'Prolonged QT interval',
 'contact dermatitis',
 'Meniere disease',
 'metabolic syndrome',
 'diabetes mellitus',
 'coronary artery disease',
 'Truncus arteriosus',
 'type 2 diabetes mellitus',
 'congenital heart defects, multiple types, 9',
 'Moebius syndrome',
 'Parkinson disease',
 'Hodgkins lymphoma',
 'genetic disorder',
 'neurodevelopmental disorder with speech delay and variable ocular anomalies',
 'cataract',
 'obesity',
 'cleft lip',

In [13]:
hematological_disorders = ['Abnormal bleeding',
'Abnormal erythrocyte morphology',
'Abnormal radial ray morphology',
'aceruloplasminemia',
'acute myeloid leukemia',
'agammaglobulinemia',
'agammaglobulinemia 10, autosomal dominant',
'agammaglobulinemia 8, autosomal dominant',
'agammaglobulinemia 8b, autosomal recessive',
'alpha-thalassemia-X-linked thrombocytopenia syndrome',
'Anemia',
'Anemia of inadequate production',
'aplasticanemia',
'atypical hemolytic-uremic syndrome',
'atypical hemolytic-uremic syndrome with H factor anomaly',
'atypical hemolytic-uremic syndrome with I factor anomaly',
'autoimmune lymphoproliferative syndrome due to CTLA4 haploinsufficiency',
'Autoimmune lymphoproliferative syndrome with recurrent viral infections',
'autoimmune lymphoproliferative syndrome type 2B',
'autoinflammatory-pancytopenia syndrome due to DNASE2 deficiency',
'autosomal dominant macrothrombocytopenia',
'B-cell acute lymphoblastic leukemia',
'B-cell immunodeficiency, distal limb anomalies, and urogenital malformations',
'beta-thalassemia-X-linked thrombocytopenia syndrome',
'Blackfan-Diamond anemia',
'bleeding diathesis due to thromboxane synthesis deficiency',
'blood coagulation disease',
'blood platelet disease',
'chronic lymphocytic leukemia',
'chronic mucocutaneous candidosis',
'chronic mucocutaneous candidiasis',
'chronic myeloproliferative disorder',
'clonal hematopoiesis',
'combined immunodeficiency',
'combined immunodeficiency due to STK4 deficiency',
'combined immunodeficiency due to ZAP70 deficiency',
'common variable immunodeficiency',
'complement factor H deficiency',
'congenital dyserythropoietic anemia type 4',
'Congenital dyserythropoietic anemia type IV',
'Congenital erythropoietic porphyria',
'deficiency anemia',
'Diamond-Blackfan anemia',
'Dorfman-Chanarin disease',
'dyskeratosis congenita',
'Fanconi anemia',
'Fanconi anemia complementation group A',
'Fanconi anemia, complementation group S',
'familial hemolytic anemia',
'Familial hemophagocytic lymphohistiocytosis',
'GATA1-Related X-Linked Cytopenia',
'Glanzmann thrombasthenia',
'Glanzmann thrombasthenia 1',
'hematologic disease',
'hematopoietic and lymphoid cell neoplasm',
'hematopoietic and lymphoid system neoplasm',
'heme oxygenase 1 deficiency',
'hemangioma',
'hemolytic anemia due to adenylate kinase deficiency',
'hemolytic anemia due to erythrocyte adenosine deaminase overproduction',
'hemorrhagic disease',
'Hermansky-Pudlak syndrome',
'Hermansky-Pudlak syndrome 3',
'Hodgkins lymphoma',
'ICF syndrome',
'Immunodeficiency with natural-killer cell deficiency and adrenal insufficiency',
'immunodeficiency 53',
'immunodeficiency 72 with autoinflammation',
'immunodeficiency 104',
'immunodeficiency 105',
'immunodeficiency disease',
'immunodeficiency, common variable, 10',
'inherited bleeding disorder, platelet-type',
'Iron deficiency anemia',
'isolated agammaglobulinemia',
'Juvenile Myelomonocytic Leukemia',
'leukemia',
'leukopenia',
'lymphangioma',
'lymphoid leukemia',
'lymphoproliferative syndrome',
'Macrothrombocytopenia',
'macrothrombocytopenia and granulocyte inclusions with or without nephritis or sensorineural hearing loss',
'megaloblastic anemia',
'mucocutaneous lymph node syndrome',
'multiple myeloma',
'myelodysplastic syndrome',
'myeloperoxidase deficiency',
'myeloproliferative disorder',
'neutropenia',
'neutropenia, severe congenital, 1, autosomal dominant',
'Noonan syndrome',
'Noonan syndrome and Noonan-related syndrome',
'Noonan syndrome-like disorder with juvenile myelomonocytic leukemia',
'non-Hodgkins lymphoma',
'Impaired ADP-induced platelet aggregation',
'platelet-type bleeding disorder 8',
'Omenn syndrome',
'pernicious anemia',
'platelet-type bleeding disorder 15',
'primary thrombocytopenia',
'Radio-ulnar synostosis - amegakaryocytic thrombocytopenia',
'Rh deficiency syndrome',
'severe combined immunodeficiency',
'severe combined immunodeficiency due to CORO1A deficiency',
'systemic inflammatory response syndrome',
'T-B+ severe combined immunodeficiency',
'T-B+ severe combined immunodeficiency due to CD45 deficiency',
'T-B+ severe combined immunodeficiency due to JAK3 deficiency',
'TFRC-related combined immunodeficiency',
'thrombocytopenia 2',
'thrombocytopenia 3',
'thrombocytopenia 4',
'thrombocytopenia 7',
'thrombocytopenia, X-linked, with or without dyserythropoietic anemia',
'Thrombocytopenia',
'thrombophilia',
'von Willebrand factor quality',
'X-linked dyserythropoetic anemia with abnormal platelets and neutropenia',
'X-linked lymphoproliferative syndrome',
'X-linked lymphoproliferative disease',
'immunodeficiency 95',
'IFIH1-related type 1 interferonopathy',
'immunodeficiency 31B',
'autoimmune lymphoproliferative syndrome',
'Abnormality of blood and blood-forming tissues',
'hereditary neutrophilia',
'autosomal recessive severe congenital neutropenia due to CSF3R deficiency',
'constitutional neutropenia',
'WHIM syndrome',
'immunodeficiency 102',
'Quebec platelet disorder',
'immunodeficiency 91 and hyperinflammation',
'Factor XIII subunit A deficiency',
'thrombophilia due to thrombin defect',
'factor XIII, A subunit, deficiency of',
'congenital factor XIII deficiency',
'immunodeficiency 117',
'macrocytic anemia',
'Shwachman-Diamond syndrome',
'aplastic anemia',
'properdin deficiency, X-linked',
'Properdin deficiency',
'thrombocytopenia 11 with multiple congenital anomalies and dysmorphic facies',
'acquired thrombocytopenia',
'Immunodeficiency',
'sea-blue histiocyte syndrome',
'Sea-blue histiocytosis',
'Jacobsen syndrome',
'Subcutaneous Panniculitis-Like T-Cell Lymphoma',
'Thalassemia',
'platelet-type bleeding disorder 10',
'neonatal anemia',
'congenital dyserythropoietic anemia type 3',
'Congenital dyserythropoietic anemia type III',
'T-cell acute lymphoblastic leukemia',
'acute lymphoblastic leukemia',
'thrombocytopenia 5',
'Absence of circulating granulocytes',
'bone marrow neoplasm',
'Fanconi anemia complementation group D1',
'lymphoid neoplasm',
'myeloid leukemia',
'leukocyte disorder',
'MHC class II deficiency',
'MHC class II deficiency 3',
'Immunodeficiency by defective expression of HLA class 2',
'polyglucosan body myopathy 1 with or without immunodeficiency',
'Chediak-Higashi syndrome',
'Chédiak-Higashi syndrome',
'Cutaneous Follicular Lymphoma',
'follicular lymphoma',
'agammaglobulinemia 7, autosomal recessive',
'immunodeficiency 36',
'activated PI3K-delta syndrome',
'anemia, nonspherocytic hemolytic, due to G6PD deficiency',
'gamma chain deficiency',
'X-Linked Combined Immunodeficiency Diseases',
'leukocyte adhesion deficiency 3',
'leukocyte adhesion deficiency',
'Leukocyte adhesion deficiency type III',
'childhood acute lymphoblastic leukemia',
'severe combined immunodeficiency due to CARMIL2 deficiency',
'immunodeficiency, common variable, 4',
'recurrent infections associated with rare immunoglobulin isotypes deficiency',
'immunodeficiency 119',
'leukocyte adhesion deficiency 1',
'Leukocyte adhesion deficiency type I',
'congenital progressive bone marrow failure-B-cell immunodeficiency-skeletal dysplasia syndrome',
'neutropenia, severe congenital, 2, autosomal dominant',
'hemolytic anemia',
'Congenital hemolytic anemia',
'Pyropoikilocytosis',
'elliptocytosis 2',
'pyropoikilocytosis, hereditary',
'hereditary spherocytosis type 3',
'hereditary spherocytosis',
'braddock-carey syndrome 2',
'platelet storage pool deficiency',
'platelet-type bleeding disorder 17',
'Platelet storage pool disease',
'lymphopenia',
'combined immunodeficiency with skin granulomas',
'severe combined immunodeficiency, autosomal recessive, T cell-negative, B cell-negative, NK cell-positive',
'combined immunodeficiency due to partial RAG1 deficiency',
'T-B- severe combined immunodeficiency',
'Combined immunodeficiency T+ B+ due to partial RAG1 deficiency',
'Immunodeficiency by defective expression of HLA class 1',
'immunodeficiency 19',
'bleeding disorder, platelet-type, 25',
'MHC class I deficiency',
'T-cell immunodeficiency with epidermodysplasia verruciformis',
'STAT3 gain of function',
'T-B+ severe combined immunodeficiency due to IL-7Ralpha deficiency',
'acquired aplastic anemia',
'ebv-positive nodal t- and nk-cell lymphoma',
'Common Hematopoietic Neoplasm',
'chronic myelomonocytic leukemia',
'adult acute myeloid leukemia',
'Abnormal leukocyte morphology',
'monocytic leukemia',
'immunodeficiency 75',
'thrombocytopenic purpura',
'Bernard-Soulier syndrome',
'polycythemia',
'pseudo-TORCH syndrome 3',
'combined immunodeficiency with faciooculoskeletal anomalies',
'immunodeficiency 14b, autosomal recessive',
'Combined immunodeficiency with facio-oculo-skeletal anomalies',
'combined immunodeficiency due to MALT1 deficiency',
'lymphoma',
'rag2 deficiency',
'recombinase activating gene 2 deficiency',
'Von Willebrand disease',
'dyskeratosis congenita, digenic',
'ataxia-pancytopenia syndrome',
'monosomy 7 myelodysplasia and leukemia syndrome 1',
'immunodeficiency 51',
'monoclonal gammopathy',
'deafness-lymphedema-leukemia syndrome',
'monocytopenia with susceptibility to infections',
'GATA2 deficiency with susceptibility to MDS/AML',
'leukemia, acute myeloid, susceptibility to',
'Deafness - lymphedema - leukemia',
'severe combined immunodeficiency due to LCK deficiency',
'platelet-type von Willebrand disease',
'Reduced antithrombin III activity',
'hereditary antithrombin deficiency',
'Anemia, Hemolytic, Autoimmune',
'autoimmune thrombocytopenia',
'autoinflammatory syndrome with immunodeficiency',
'immunodeficiency 39',
'pancytopenia due to IKZF1 mutations',
'primary familial polycythemia due to EPO receptor mutation',
'Heinz body anemia',
'alpha thalassemia spectrum',
'hemoglobin H disease',
'beta thalassemia',
'Alpha-thalassemia',
'cyclic hematopoiesis',
'X-linked severe congenital neutropenia',
'Cyclic neutropenia',
'immunodeficiency 77',
'recurrent Neisseria infections due to factor D deficiency',
'familial cold autoinflammatory syndrome 3',
'autoinflammation-PLCG2-associated antibody deficiency-immune dysregulation',
'PLCG2-associated antibody deficiency and immune dysregulation',
'immunodeficiency 33',
'immunodeficiency 67',
'Immunodeficiency due to interleukin-1 receptor-associated kinase-4 deficiency',
'severe combined immunodeficiency due to CARD11 deficiency',
'BENTA disease',
'combined immunodeficiency due to LRBA deficiency',
'bone marrow failure syndrome 6',
'thrombocytopenia, anemia, and myelofibrosis',
'monosomy 7 myelodysplasia and leukemia syndrome 2',
'Autosomal dominant methemoglobinemia',
'severe combined immunodeficiency due to LAT deficiency',
'Abnormal hemoglobin',
'Persistence of hemoglobin F',
'Reduced beta/alpha synthesis ratio',
'beta-thalassemia HBB/LCRB',
'delta-beta-thalassemia',
'immune deficiency, familial variable',
'immunodeficiency, common variable, 2',
'hyper-IgM syndrome type 2',
'hyper-IgM syndrome type 5', 
'hyper-IgM syndrome type 3',
'immunoglobulin A deficiency 2',
'sickle cell disease and related diseases',
'Hemoglobin SC Disease',
'dominant beta-thalassemia',
'sickle cell anemia',
'hemoglobin E disease',
'beta-thalassemia major',
'beta-thalassemia intermedia',
'hemoglobin E-beta-thalassemia syndrome',
'sickle cell-hemoglobin c disease syndrome',
'hemoglobin M disease',
'hemoglobin D disease',
'hemoglobinopathy',
'erythrocytosis, familial, 6',
'Hemoglobin E - beta-thalassemia',
'Sickle cell - hemoglobin C disease',
'Beta-thalassemia',
'childhood acute myeloid leukemia',
'Prolonged bleeding time',
'fetal and neonatal alloimmune thrombocytopenia',
'bleeding disorder, platelet-type, 24',
'Paraproteinemia',
'Aicardi-Goutieres syndrome', 
'Aicardi-Goutières syndrome',
'Aicardi-Goutieres syndrome 7',
'Omenn syndrome',
'CHARGE syndrome',
'chronic granulomatous disease',
'pyogenic granuloma',
'CINCA syndrome',
'Muckle-Wells syndrome',
'pulmonary alveolar proteinosis with hypogammaglobulinemia',
'Congenital pulmonary alveolar proteinosis',
'Susceptibility to viral and mycobacterial infections',
'ectodermal dysplasia and immunodeficiency 2',
'Hennekam lymphangiectasia-lymphedema syndrome 2',
'hyper-IgE recurrent infection syndrome 4A, autosomal dominant',
'Autosomal recessive hyper-IgE syndrome',
'hyper-IgE recurrent infection syndrome 5, autosomal recessive',
'hyper-IgE syndrome 6, autosomal dominant, with recurrent infections',
'hyper-IgE recurrent infection syndrome 1, autosomal dominant',
'hyper-IgE syndrome', 'Autosomal dominant hyper-IgE syndrome',
'immunodeficiency-centromeric instability-facial anomalies syndrome',
'Li-Fraumeni syndrome',
'LIG4 syndrome',
'MIRAGE syndrome',
'overhydrated hereditary stomatocytosis',
'purine nucleoside phosphorylase deficiency',
'STAT3 gain of function',
'trypanosomiasis'
]
hematological_df = ot_df[ot_df['disease_name'].isin(hematological_disorders)]

In [14]:
synonym_map = {
    # Aicardi–Goutières
    'Aicardi-Goutieres syndrome': 'Aicardi-Goutières Syndrome',
    'Aicardi-Goutieres syndrome 7': 'Aicardi-Goutières Syndrome',

    # Aplastic anemia
    'aplasticanemia': 'Aplastic Anemia',
    'acquired aplastic anemia': 'Aplastic Anemia',

    # Diamond–Blackfan anemia
    'Blackfan-Diamond anemia': 'Diamond-Blackfan Anemia',

    # Congenital dyserythropoietic anemia
    'Congenital dyserythropoietic anemia type 4': 'Congenital Dyserythropoietic Anemia Type IV',
    'Congenital dyserythropoietic anemia type 3': 'Congenital Dyserythropoietic Anemia Type III',

    # Hereditary spherocytosis
    'hereditary spherocytosis type 3': 'Hereditary Spherocytosis',

    # Pyropoikilocytosis
    'pyropoikilocytosis, hereditary': 'Pyropoikilocytosis',

    # Chronic mucocutaneous candidiasis
    'chronic mucocutaneous candidosis': 'Chronic Mucocutaneous Candidiasis',

    # Chediak–Higashi syndrome
    'Chédiak-Higashi syndrome': 'Chediak-Higashi Syndrome',

    # Hemoglobin E–beta thalassemia
    'Hemoglobin E - beta-thalassemia': 'Hemoglobin E-Beta-Thalassemia Syndrome',

    # Sickle cell / HbC disease
    'sickle cell - hemoglobin C disease': 'Sickle Cell–Hemoglobin C Disease Syndrome',

    # Von Willebrand disease
    'von Willebrand factor quality': 'Von Willebrand Disease',
    'platelet-type von Willebrand disease': 'Von Willebrand Disease',

    # Platelet-type bleeding disorders
    'bleeding disorder, platelet-type, 24': 'Bleeding Disorder, Platelet-Type 24',
    'bleeding disorder, platelet-type, 25': 'Bleeding Disorder, Platelet-Type 25',

    # Platelet storage pool disease
    'platelet storage pool deficiency': 'Platelet Storage Pool Disease',

    # Sea-blue histiocytosis
    'sea-blue histiocyte syndrome': 'Sea-Blue Histiocytosis',

    # Thalassemias
    'alpha thalassemia spectrum': 'Alpha-Thalassemia',
    'beta thalassemia': 'Beta-Thalassemia',
    'dominant beta-thalassemia': 'Beta-Thalassemia',
    'beta-thalassemia major': 'Beta-Thalassemia',
    'beta-thalassemia intermedia': 'Beta-Thalassemia',
    'beta-thalassemia HBB/LCRB': 'Beta-Thalassemia',
    'delta-beta-thalassemia': 'Beta-Thalassemia',

    # Glanzmann
    'Glanzmann thrombasthenia 1': 'Glanzmann Thrombasthenia',

    # Hemoglobin variants
    'hemoglobin E disease': 'Hemoglobinopathy',
    'hemoglobin D disease': 'Hemoglobinopathy',
    'hemoglobin M disease': 'Hemoglobinopathy',
    'Hemoglobin SC Disease': 'Hemoglobinopathy',
    'hemoglobinopathy': 'Hemoglobinopathy',

    # Lymphomas
    'non-Hodgkins lymphoma': 'Non-Hodgkin Lymphoma',
    'Hodgkins lymphoma': 'Hodgkin Lymphoma',

    # MHC II deficiency
    'MHC class II deficiency 3': 'MHC Class II Deficiency',

    # Factor XIII
    'factor XIII, A subunit, deficiency of': 'Factor XIII Subunit A Deficiency',
    'congenital factor XIII deficiency': 'Factor XIII Subunit A Deficiency',


    # Leukocyte adhesion deficiency
    'leukocyte adhesion deficiency 1': 'Leukocyte Adhesion Deficiency Type I',

    # Complement
    'complement factor H deficiency': 'Complement Factor H Deficiency',

    # Myeloperoxidase deficiency (note: your original synonym mapped incorrectly)
    'myeloperoxidase deficiency': 'Myeloperoxidase Deficiency',

    # RAG2 deficiency
    'rag2 deficiency': 'RAG2 Deficiency',
    'recombinase activating gene 2 deficiency': 'RAG2 Deficiency',

    # GATA1-related cytopenia
    'X-linked dyserythropoetic anemia with abnormal platelets and neutropenia' : 'GATA1-related cytopenia',
    'GATA1-Related X-Linked Cytopenia': 'GATA1-related cytopenia',

    # X-linked lymphoproliferative syndrome
    'X-linked lymphoproliferative syndrome': 'X-linked lymphoproliferative disease',
    'X-linked lymphoproliferative disease': 'X-linked lymphoproliferative disease'
}


In [15]:
import re
import string

# Words that stay lowercase in medical title case formatting
LOWERCASE_EXCEPTIONS = {
    "of", "and", "or", "in", "with", "without", "due", "to",
    "the", "a", "an", "by", "for", "type", "types", "from"
}

def normalize_spacing(text: str) -> str:
    """Remove extra spaces, fix spacing around hyphens, unify punctuation."""
    text = text.strip()

    # Normalize hyphens with consistent single spacing
    text = re.sub(r"\s*-\s*", " - ", text)

    # Collapse multiple spaces → one
    text = re.sub(r"\s+", " ", text)

    return text


def normalize_punctuation(text: str) -> str:
    """Standardize punctuation and unicode variants."""
    text = text.replace("–", "-")  # en dash → hyphen
    text = text.replace("—", "-")  # em dash → hyphen
    text = text.replace("–", "-")
    text = text.replace("’", "'")
    text = text.replace("é", "e")  # handle Aicardi-Goutières syndrome variants
    return text

def safe_title_case(text: str) -> str:
    """
    Capitalize only the first character of each word.
    Do NOT change any other characters (preserves gene symbols, roman numerals, acronyms).
    """
    words = text.split()
    result = []
    for w in words:
        if len(w) == 0:
            result.append(w)
        elif w.lower() in LOWERCASE_EXCEPTIONS:
            result.append(w.lower())
        else:
            # uppercase first character, leave rest unchanged
            result.append(w[0].upper() + w[1:])
    return " ".join(result)


def standardize_disease_name(name: str, synonym_map=None) -> str:
    """Normalize, title-case, apply synonyms, spacing, punctuation."""
    if not isinstance(name, str):
        return name

    # Apply synonym mapping first if provided
    if synonym_map and name in synonym_map:
        name = synonym_map[name]

    # 1. normalize punctuation
    name = normalize_punctuation(name)

    # 2. normalize spacing
    name = normalize_spacing(name)

    # 3. convert to medical Title Case
    name = safe_title_case(name)

    return name


In [16]:
hematological_df["disease_name"]

933                          complement factor H deficiency
939       atypical hemolytic-uremic syndrome with I fact...
940       atypical hemolytic-uremic syndrome with I fact...
941       atypical hemolytic-uremic syndrome with I fact...
942                      atypical hemolytic-uremic syndrome
                                ...                        
184847                             Glanzmann thrombasthenia
184854                Autosomal dominant hyper-IgE syndrome
184855                Autosomal dominant hyper-IgE syndrome
185557                                    lymphoid leukemia
185558                                    lymphoid leukemia
Name: disease_name, Length: 6739, dtype: object

In [17]:
# string manipulation to clean disease names
hematological_df["disease_name_clean"] = hematological_df["disease_name"].apply(
    lambda x: standardize_disease_name(x, synonym_map)
)


/tmp/ipykernel_2749781/4153216346.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hematological_df["disease_name_clean"] = hematological_df["disease_name"].apply(


In [18]:
hematological_df[["disease_name_clean", "disease_name"]].head(100)

,disease_name_clean,disease_name
933,Complement Factor H Deficiency,complement factor H deficiency
939,Atypical Hemolytic - Uremic Syndrome with I Fa...,atypical hemolytic-uremic syndrome with I fact...
940,Atypical Hemolytic - Uremic Syndrome with I Fa...,atypical hemolytic-uremic syndrome with I fact...
941,Atypical Hemolytic - Uremic Syndrome with I Fa...,atypical hemolytic-uremic syndrome with I fact...
942,Atypical Hemolytic - Uremic Syndrome,atypical hemolytic-uremic syndrome
...,...,...
2314,Autosomal Dominant Macrothrombocytopenia,autosomal dominant macrothrombocytopenia
2315,Autosomal Dominant Macrothrombocytopenia,autosomal dominant macrothrombocytopenia
2316,Glanzmann Thrombasthenia,Glanzmann thrombasthenia 1
2317,Glanzmann Thrombasthenia,Glanzmann thrombasthenia 1


In [19]:
hematological_df[hematological_df.disease_name_clean == 'GATA1 - Related Cytopenia']

,Unnamed: 0,ensembl_id,genes,cluster,disease_id,disease_name,therapeutic_area_name,therapeutic_area_id,association_score,disease_ancestors,chembl,eva,gene_burden,genomics_england,gwas_credible_sets,disease_name_clean
31561,31561,ENSG00000102145,GATA1,Early megakaryocyte - erythroid differentiation,MONDO_0010444,X-linked dyserythropoetic anemia with abnormal...,"genetic, familial or congenital disease",OTAR_0000018,0.727418,"MONDO_0100089, MONDO_0002280, OTAR_0000018, MO...",NaN,0.856870,NaN,0.928567,NaN,GATA1 - Related Cytopenia
31562,31562,ENSG00000102145,GATA1,Early megakaryocyte - erythroid differentiation,MONDO_0010444,X-linked dyserythropoetic anemia with abnormal...,hematologic disease,EFO_0005803,0.727418,"MONDO_0100089, MONDO_0002280, OTAR_0000018, MO...",NaN,0.856870,NaN,0.928567,NaN,GATA1 - Related Cytopenia
31571,31571,ENSG00000102145,GATA1,Early megakaryocyte - erythroid differentiation,MONDO_0100089,GATA1-Related X-Linked Cytopenia,"genetic, familial or congenital disease",OTAR_0000018,0.438716,"EFO_0000508, EFO_0005803, OTAR_0000018",NaN,0.909794,NaN,NaN,NaN,GATA1 - Related Cytopenia
31572,31572,ENSG00000102145,GATA1,Early megakaryocyte - erythroid differentiation,MONDO_0100089,GATA1-Related X-Linked Cytopenia,hematologic disease,EFO_0005803,0.438716,"EFO_0000508, EFO_0005803, OTAR_0000018",NaN,0.909794,NaN,NaN,NaN,GATA1 - Related Cytopenia


In [20]:
consolidated_disease_map = {
    # --- Red cell / hemoglobin / thalassemia / sickle cell -----------------

    # Thalassemias
    "Alpha - Thalassemia": "Alpha-Thalassemia",
    "Beta - Thalassemia": "Beta-Thalassemia",
    "Beta - Thalassemia - X - Linked Thrombocytopenia Syndrome": "Beta-Thalassemia X-Linked Thrombocytopenia Syndrome",
    "Reduced Beta/alpha Synthesis Ratio": "Beta-Thalassemia",
    "Persistence of Hemoglobin F": "Beta-Thalassemia",  # fetal Hb persistence in thal phenotype
    "Hemoglobin E - Beta - Thalassemia Syndrome": "Beta-Thalassemia",
    "Thalassemia": "Thalassemia",
    "Erythrocytosis, Familial, 6": "Polycythemia",
    "Polycythemia": "Polycythemia",
    "Primary Familial Polycythemia due to EPO Receptor Mutation": "Polycythemia",

    # Hemoglobin variants / abnormalities
    "Abnormal Hemoglobin": "Hemoglobinopathy",
    "Hemoglobinopathy": "Hemoglobinopathy",

    # Sickle cell spectrum
    "Sickle Cell Anemia": "Sickle Cell Disease",
    "Sickle Cell Disease and Related Diseases": "Sickle Cell Disease",
    "Sickle Cell - Hemoglobin C Disease": "Sickle Cell Disease",
    "Sickle Cell - Hemoglobin C Disease Syndrome": "Sickle Cell Disease",

    # Hemolytic anemias
    "Hemolytic Anemia": "Hemolytic Anemia",
    "Hemolytic Anemia due to Adenylate Kinase Deficiency": "Hemolytic Anemia",
    "Hemolytic Anemia due to Erythrocyte Adenosine Deaminase Overproduction": "Hemolytic Anemia",

    # Atypical Hemolytic Uremic Syndrome
    "Atypical Hemolytic - Uremic Syndrome": "Atypical Hemolytic Uremic Syndrome",
    "Atypical Hemolytic - Uremic Syndrome with H Factor Anomaly": "Atypical Hemolytic Uremic Syndrome",
    "Atypical Hemolytic - Uremic Syndrome with I Factor Anomaly": "Atypical Hemolytic Uremic Syndrome",


    # Congenital dyserythropoietic anaemias
    "Congenital Dyserythropoietic Anemia type 3": "Congenital Dyserythropoietic Anemia",
    "Congenital Dyserythropoietic Anemia type III": "Congenital Dyserythropoietic Anemia",
    "Congenital Dyserythropoietic Anemia type 4": "Congenital Dyserythropoietic Anemia",
    "Congenital Dyserythropoietic Anemia type IV": "Congenital Dyserythropoietic Anemia",

    

    # Aplastic / deficiency anaemia (leave mostly distinct)
    "Deficiency Anemia": "Deficiency Anemia",
    "Anemia of Inadequate Production": "Anemia of Inadequate Production",
    "Aplastic Anemia": "Aplastic Anemia",
    "Neonatal Anemia": "Neonatal Anemia",


    # --- Platelet / coagulation / bleeding ---------------------------------

    # Platelet-type bleeding disorders
    "Bleeding Disorder, Platelet - type 24": "Platelet-Type Bleeding Disorder",
    "Bleeding Disorder, Platelet - type 25": "Platelet-Type Bleeding Disorder",
    "Inherited Bleeding Disorder, Platelet - type": "Platelet-Type Bleeding Disorder",
    "Platelet - type Bleeding Disorder 8": "Platelet-Type Bleeding Disorder",
    "Platelet - type Bleeding Disorder 10": "Platelet-Type Bleeding Disorder",
    "Platelet - type Bleeding Disorder 15": "Platelet-Type Bleeding Disorder",
    "Platelet - type Bleeding Disorder 17": "Platelet-Type Bleeding Disorder",

    # Platelet storage pool
    "Platelet Storage Pool Disease": "Platelet Storage Pool Disease",

    # Thrombophilias (keep distinct but unify small variants)
    "Thrombophilia": "Thrombophilia",
    "Thrombophilia due to Thrombin Defect": "Thrombophilia",

    # Thrombocytopenias (keep distinct but unify small variants)
    "Acquired Thrombocytopenia": "Thrombocytopenia",
    "Primary Thrombocytopenia": "Thrombocytopenia",
    "Thrombocytopenia": "Thrombocytopenia",
    "Thrombocytopenia": "Thrombocytopenia",
    "Thrombocytopenia 2": "Thrombocytopenia",
    "Thrombocytopenia 3": "Thrombocytopenia",
    "Thrombocytopenia 4": "Thrombocytopenia",
    "Thrombocytopenia 5": "Thrombocytopenia",
    "Thrombocytopenia 7": "Thrombocytopenia",
    "Thrombocytopenia 11 with Multiple Congenital Anomalies and Dysmorphic Facies": "Thrombocytopenia",
    "Thrombocytopenic Purpura": "Thrombocytopenia",
    "Thrombocytopenia, X - Linked, with or without Dyserythropoietic Anemia": "Thrombocytopenia",
    "Autoimmune Thrombocytopenia": "Thrombocytopenia",

    # Macrothrombocytopenia cluster
    "Macrothrombocytopenia": "Macrothrombocytopenia",
    "Autosomal Dominant Macrothrombocytopenia": "Macrothrombocytopenia",
    "Macrothrombocytopenia and Granulocyte Inclusions with or without Nephritis or Sensorineural Hearing Loss":
        "Macrothrombocytopenia",


    # Platelet disease generic
    "Blood Platelet Disease": "Platelet Disorder",

    # Coagulation / bleeding
    "Abnormal Bleeding": "Abnormal Bleeding",
    "Prolonged Bleeding Time": "Prolonged Bleeding Time",
    "Blood Coagulation Disease": "Blood Coagulation Disorder",
    "Hemorrhagic Disease": "Hemorrhagic Disease",
    "Bleeding Diathesis due to Thromboxane Synthesis Deficiency": "Thromboxane Synthesis Defect",
    "Von Willebrand Disease": "Von Willebrand Disease",
    "Impaired ADP - Induced Platelet Aggregation": "Impaired ADP-Induced Platelet Aggregation",

    # ---- Coagulation / bleeding phenotypes ----
    "Factor VII Measurement": "Coagulation Phenotype",
    "Factor VIII Measurement": "Coagulation Phenotype",
    "Factor XI Measurement": "Coagulation Phenotype",

    # --- Myeloid / neutrophil / marrow failure -----------------------------

    "Cyclic Hematopoiesis": "Cyclic Neutropenia",
    "Cyclic Neutropenia": "Cyclic Neutropenia",
    "Constitutional Neutropenia": "Constitutional Neutropenia",
    "Neutropenia": "Neutropenia",
    "Neutropenia, Severe Congenital, 1, Autosomal Dominant": "Severe Congenital Neutropenia",
    "Neutropenia, Severe Congenital, 2, Autosomal Dominant": "Severe Congenital Neutropenia",
    "X - Linked Severe Congenital Neutropenia": "Severe Congenital Neutropenia",
    "Autosomal Recessive Severe Congenital Neutropenia due to CSF3R Deficiency":
        "Severe Congenital Neutropenia",
    "Constitutional Neutropenia": "Neutropenia",

    "Myeloproliferative Disorder": "Myeloproliferative Disorder",
    "Myelodysplastic Syndrome": "Myelodysplastic Syndrome",
    "Chronic Myeloproliferative Disorder": "Myeloproliferative Disorder",
    "Chronic Myelomonocytic Leukemia": "Chronic Myelomonocytic Leukemia",

    "Bone Marrow Failure Syndrome 6": "Bone Marrow Failure Syndrome",
    "Fanconi Anemia": "Fanconi Anemia",
    "Fanconi Anemia Complementation Group a": "Fanconi Anemia",
    "Fanconi Anemia Complementation Group D1": "Fanconi Anemia",
    "Fanconi Anemia, Complementation Group S": "Fanconi Anemia",
    'GATA1 - Related Cytopenia':'GATA1-Related Cytopenia',

    "Shwachman - Diamond Syndrome": "Shwachman-Diamond Syndrome",
    "Dyskeratosis Congenita": "Dyskeratosis Congenita",
    "Dyskeratosis Congenita, Digenic": "Dyskeratosis Congenita",

    "Bone Marrow Neoplasm": "Bone Marrow Neoplasm",
    "Common Hematopoietic Neoplasm": "Common Hematopoietic Neoplasm",

    # --- Leukemia / lymphoma / plasma cell disorders -----------------------

    # AML
    "Acute Myeloid Leukemia": "Acute Myeloid Leukemia",
    "Adult Acute Myeloid Leukemia": "Acute Myeloid Leukemia",
    "Childhood Acute Myeloid Leukemia": "Acute Myeloid Leukemia",
    "Leukemia, Acute Myeloid, Susceptibility to": "Acute Myeloid Leukemia",

    # ALL
    "Acute Lymphoblastic Leukemia": "Acute Lymphoblastic Leukemia",
    "Childhood Acute Lymphoblastic Leukemia": "Acute Lymphoblastic Leukemia",
    "B - Cell Acute Lymphoblastic Leukemia": "B-Cell Acute Lymphoblastic Leukemia",
    "T - Cell Acute Lymphoblastic Leukemia": "T-Cell Acute Lymphoblastic Leukemia",
    "Monosomy 7 Myelodysplasia and Leukemia Syndrome 1": "Monosomy 7 Myelodysplasia and Leukemia Syndrome",
    "Monosomy 7 Myelodysplasia and Leukemia Syndrome 2": "Monosomy 7 Myelodysplasia and Leukemia Syndrome",

    # Other leukemias
    "Juvenile Myelomonocytic Leukemia": "Juvenile Myelomonocytic Leukemia",
    "Monocytic Leukemia": "Monocytic Leukemia",
    "Lymphoid Leukemia": "Lymphoid Leukemia",
    "Leukemia": "Leukemia",

    # Lymphoma spectrum
    "Lymphoma": "Lymphoma",
    "Hodgkin Lymphoma": "Hodgkin Lymphoma",
    "Non - Hodgkin Lymphoma": "Non-Hodgkin Lymphoma",
    "Follicular Lymphoma": "Follicular Lymphoma",
    "Cutaneous Follicular Lymphoma": "Cutaneous Follicular Lymphoma",
    "Subcutaneous Panniculitis - Like T - Cell Lymphoma": "Subcutaneous Panniculitis-Like T-Cell Lymphoma",
    "Ebv - Positive Nodal T - and Nk - Cell Lymphoma": "EBV-Positive Nodal T- and NK-Cell Lymphoma",




    # Mixed neoplasm descriptors
    "Hematopoietic and Lymphoid Cell Neoplasm": "Hematopoietic and Lymphoid Neoplasm",
    "Hematopoietic and Lymphoid System Neoplasm": "Hematopoietic and Lymphoid Neoplasm",


    # --- Immunodeficiency (SCID, CID, CVID, agammaglobulinemia, etc.) ------

    # Generic immunodeficiency
    "Immunodeficiency": "Immunodeficiency, Unspecified",
    "Immunodeficiency Disease": "Immunodeficiency, Unspecified",
    "Immune Deficiency, Familial Variable": "Immunodeficiency, Familial Variable",

    # CVID
    "Common Variable Immunodeficiency": "Common Variable Immunodeficiency",
    "Immunodeficiency, Common Variable, 2": "Common Variable Immunodeficiency",
    "Immunodeficiency, Common Variable, 4": "Common Variable Immunodeficiency",
    "Immunodeficiency, Common Variable, 10": "Common Variable Immunodeficiency",

    # Agammaglobulinemias
    "Agammaglobulinemia": "Agammaglobulinemia",
    "Agammaglobulinemia 7, Autosomal Recessive": "Agammaglobulinemia",
    "Agammaglobulinemia 8, Autosomal Dominant": "Agammaglobulinemia",
    "Agammaglobulinemia 8b, Autosomal Recessive": "Agammaglobulinemia",
    "Agammaglobulinemia 10, Autosomal Dominant": "Agammaglobulinemia",
    "Isolated Agammaglobulinemia": "Agammaglobulinemia",

    # IgA deficiency
    "Immunoglobulin a Deficiency 2": "Immunoglobulin A Deficiency",

    # Hyper-IgE syndromes
    "Autosomal Dominant Hyper - IgE Syndrome": "Hyper-IgE Syndrome",
    "Autosomal Recessive Hyper - IgE Syndrome": "Hyper-IgE Syndrome",
    "Hyper - IgE Syndrome": "Hyper-IgE Syndrome",
    "Hyper - IgE Recurrent Infection Syndrome 1, Autosomal Dominant": "Hyper-IgE Syndrome",
    "Hyper - IgE Recurrent Infection Syndrome 4A, Autosomal Dominant": "Hyper-IgE Syndrome",
    "Hyper - IgE Recurrent Infection Syndrome 5, Autosomal Recessive": "Hyper-IgE Syndrome",
    "Hyper - IgE Syndrome 6, Autosomal Dominant, with Recurrent Infections": "Hyper-IgE Syndrome",

    # Hyper-IgM syndromes
    "Hyper - IgM Syndrome type 2": "Hyper-IgM Syndrome",
    "Hyper - IgM Syndrome type 3": "Hyper-IgM Syndrome",
    "Hyper - IgM Syndrome type 5": "Hyper-IgM Syndrome",

    # Combined immunodeficiencies (non-SCID)
    "Combined Immunodeficiency": "Combined Immunodeficiency",
    "Combined Immunodeficiency T+ B+ due to Partial RAG1 Deficiency": "Combined Immunodeficiency",
    "Combined Immunodeficiency due to Partial RAG1 Deficiency": "Combined Immunodeficiency",
    "Combined Immunodeficiency due to LRBA Deficiency": "Combined Immunodeficiency",
    "Combined Immunodeficiency due to MALT1 Deficiency": "Combined Immunodeficiency",
    "Combined Immunodeficiency due to STK4 Deficiency": "Combined Immunodeficiency",
    "Combined Immunodeficiency due to ZAP70 Deficiency": "Combined Immunodeficiency",
    "Combined Immunodeficiency with Facio - Oculo - Skeletal Anomalies": "Combined Immunodeficiency",
    "Combined Immunodeficiency with Faciooculoskeletal Anomalies": "Combined Immunodeficiency",
    "Combined Immunodeficiency with Skin Granulomas": "Combined Immunodeficiency",
    "TFRC - Related Combined Immunodeficiency": "Combined Immunodeficiency",
    "X - Linked Combined Immunodeficiency Diseases": "Combined Immunodeficiency",

    # “Immunodeficiency N” (numbered) – if you want to group them:
    "Immunodeficiency 102": "Monogenic Immunodeficiency",
    "Immunodeficiency 104": "Monogenic Immunodeficiency",
    "Immunodeficiency 105": "Monogenic Immunodeficiency",
    "Immunodeficiency 117": "Monogenic Immunodeficiency",
    "Immunodeficiency 119": "Monogenic Immunodeficiency",
    "Immunodeficiency 14b, Autosomal Recessive": "Monogenic Immunodeficiency",
    "Immunodeficiency 19": "Monogenic Immunodeficiency",
    "Immunodeficiency 31B": "Monogenic Immunodeficiency",
    "Immunodeficiency 33": "Monogenic Immunodeficiency",
    "Immunodeficiency 36": "Monogenic Immunodeficiency",
    "Immunodeficiency 39": "Monogenic Immunodeficiency",
    "Immunodeficiency 51": "Monogenic Immunodeficiency",
    "Immunodeficiency 53": "Monogenic Immunodeficiency",
    "Immunodeficiency 67": "Monogenic Immunodeficiency",
    "Immunodeficiency 72 with Autoinflammation": "Monogenic Immunodeficiency",
    "Immunodeficiency 75": "Monogenic Immunodeficiency",
    "Immunodeficiency 77": "Monogenic Immunodeficiency",
    "Immunodeficiency 91 and Hyperinflammation": "Monogenic Immunodeficiency",
    "Immunodeficiency 95": "Monogenic Immunodeficiency",

    # ---- Immunodeficiencies ----
    "Immunodeficiency by Defective Expression of HLA Class 1": "HLA Class I Deficiency",
    "Immunodeficiency by Defective Expression of HLA Class 2": "HLA Class II Deficiency",
    "Immunodeficiency, Familial Variable": "Familial Variable Immunodeficiency",
    "Immunodeficiency, Unspecified": "Immunodeficiency, Unspecified",
    "Immunodeficiency with Natural - Killer Cell Deficiency and Adrenal Insufficiency":
        "NK Cell Deficiency with Adrenal Insufficiency",
    "Immunodeficiency due to Interleukin - 1 Receptor - Associated Kinase - 4 Deficiency":
        "IRAK4 Deficiency",
    "Immunodeficiency-Centromeric Instability-Facial Anomalies Syndrome":
        "ICF Syndrome",    

    # ---- Autoinflammatory / immune dysregulation ----
    "Familial Cold Autoinflammatory Syndrome 3": "Autoinflammatory Syndrome",
    "Autoinflammatory Syndrome with Immunodeficiency": "Autoinflammatory Syndrome",
    "Autoinflammatory - Pancytopenia Syndrome due to DNASE2 Deficiency":
        "Autoinflammatory Pancytopenia Syndrome",

    # SCID family – *fully* consolidated here
    "Severe Combined Immunodeficiency": "Severe Combined Immunodeficiency",
    "T - B - Severe Combined Immunodeficiency": "Severe Combined Immunodeficiency",
    "T - B+ Severe Combined Immunodeficiency": "Severe Combined Immunodeficiency",
    "T - B+ Severe Combined Immunodeficiency due to CD45 Deficiency": "Severe Combined Immunodeficiency",
    "T - B+ Severe Combined Immunodeficiency due to IL - 7Ralpha Deficiency": "Severe Combined Immunodeficiency",
    "T - B+ Severe Combined Immunodeficiency due to JAK3 Deficiency": "Severe Combined Immunodeficiency",
    "Severe Combined Immunodeficiency due to CARD11 Deficiency": "Severe Combined Immunodeficiency",
    "Severe Combined Immunodeficiency due to CARMIL2 Deficiency": "Severe Combined Immunodeficiency",
    "Severe Combined Immunodeficiency due to CORO1A Deficiency": "Severe Combined Immunodeficiency",
    "Severe Combined Immunodeficiency due to LAT Deficiency": "Severe Combined Immunodeficiency",
    "Severe Combined Immunodeficiency due to LCK Deficiency": "Severe Combined Immunodeficiency",
    "Severe Combined Immunodeficiency, Autosomal Recessive, T Cell - Negative, B Cell - Negative, NK Cell - Positive":
        "Severe Combined Immunodeficiency",

    # ALPS / lymphoproliferative
    "Autoimmune Lymphoproliferative Syndrome": "Autoimmune Lymphoproliferative Syndrome",
    "Autoimmune Lymphoproliferative Syndrome due to CTLA4 Haploinsufficiency": "Autoimmune Lymphoproliferative Syndrome",
    "Autoimmune Lymphoproliferative Syndrome type 2B": "Autoimmune Lymphoproliferative Syndrome",
    "Autoimmune Lymphoproliferative Syndrome with Recurrent Viral Infections": "Autoimmune Lymphoproliferative Syndrome",

    # PLCG2 / WHIM etc.
    "Autoinflammation - PLCG2 - Associated Antibody Deficiency - Immune Dysregulation":
        "PLCG2-Associated Antibody Deficiency and Immune Dysregulation",
    "PLCG2 - Associated Antibody Deficiency and Immune Dysregulation":
        "PLCG2-Associated Antibody Deficiency and Immune Dysregulation",
    "WHIM Syndrome": "WHIM Syndrome",

    # LAD
    "Leukocyte Adhesion Deficiency": "Leukocyte Adhesion Deficiency",
    "Leukocyte Adhesion Deficiency 3": "Leukocyte Adhesion Deficiency",
    "Leukocyte Adhesion Deficiency type I": "Leukocyte Adhesion Deficiency",
    "Leukocyte Adhesion Deficiency type III": "Leukocyte Adhesion Deficiency",

    # Complement / properdin
    "Properdin Deficiency": "Properdin Deficiency",
    "Properdin Deficiency, X - Linked": "Properdin Deficiency",

    # Misc immune
    "Immunodeficiency - Centromeric Instability - Facial Anomalies Syndrome":
        "Immunodeficiency-Centromeric Instability-Facial Anomalies Syndrome",
    "ICF Syndrome": "Immunodeficiency-Centromeric Instability-Facial Anomalies Syndrome",


    # --- “Named” syndromes / systemic --------------------------------------

    "CHARGE Syndrome": "CHARGE Syndrome",
    "Li - Fraumeni Syndrome": "Li-Fraumeni Syndrome",
    "Noonan Syndrome": "Noonan Syndrome",
    "Noonan Syndrome and Noonan - Related Syndrome": "Noonan Syndrome",
    "Noonan Syndrome - Like Disorder with Juvenile Myelomonocytic Leukemia": "Noonan Syndrome-Like Disorder with Juvenile Myelomonocytic Leukemia",
    "Deafness - Lymphedema - Leukemia": "Deafness-Lymphedema-Leukemia Syndrome",
    "Deafness - Lymphedema - Leukemia Syndrome": "Deafness-Lymphedema-Leukemia Syndrome",
}


In [21]:

hematological_df["disease_name_clean"] = hematological_df["disease_name_clean"].apply(
    lambda x: consolidated_disease_map.get(x, x)
)

/tmp/ipykernel_2749781/3278018351.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hematological_df["disease_name_clean"] = hematological_df["disease_name_clean"].apply(


In [22]:
sorted(hematological_df["disease_name_clean"].unique().tolist())

['Abnormal Bleeding',
 'Abnormal Erythrocyte Morphology',
 'Abnormal Leukocyte Morphology',
 'Abnormal Radial Ray Morphology',
 'Abnormality of Blood and Blood - Forming Tissues',
 'Absence of Circulating Granulocytes',
 'Aceruloplasminemia',
 'Activated PI3K - Delta Syndrome',
 'Acute Lymphoblastic Leukemia',
 'Acute Myeloid Leukemia',
 'Agammaglobulinemia',
 'Aicardi - Goutières Syndrome',
 'Alpha-Thalassemia',
 'Anemia',
 'Anemia of Inadequate Production',
 'Anemia, Hemolytic, Autoimmune',
 'Anemia, Nonspherocytic Hemolytic, due to G6PD Deficiency',
 'Aplastic Anemia',
 'Ataxia - Pancytopenia Syndrome',
 'Atypical Hemolytic Uremic Syndrome',
 'Autoimmune Lymphoproliferative Syndrome',
 'Autoinflammatory Pancytopenia Syndrome',
 'Autoinflammatory Syndrome',
 'Autosomal Dominant Methemoglobinemia',
 'B - Cell Immunodeficiency, Distal Limb Anomalies, and Urogenital Malformations',
 'B-Cell Acute Lymphoblastic Leukemia',
 'BENTA Disease',
 'Bernard - Soulier Syndrome',
 'Beta-Thalassemi

In [23]:
group_map = {

    # --- RED CELL DISORDERS ---
    "Abnormal Erythrocyte Morphology": "Red Cell disorder",
    "Alpha-Thalassemia": "Red Cell disorder",
    "Anemia": "Red Cell disorder",
    "Anemia of Inadequate Production": "Red Cell disorder",
    "Anemia, Hemolytic, Autoimmune": "Red Cell disorder",
    "Anemia, Nonspherocytic Hemolytic, due to G6PD Deficiency": "Red Cell disorder",
    "Autosomal Dominant Methemoglobinemia": "Red Cell disorder",
    "Beta-Thalassemia": "Red Cell disorder",
    "Congenital Dyserythropoietic Anemia": "Red Cell disorder",
    "Congenital Erythropoietic Porphyria": "Red Cell disorder",
    "Congenital Hemolytic Anemia": "Red Cell disorder",
    "Deficiency Anemia": "Red Cell disorder",
    "Diamond - Blackfan Anemia": "Red Cell disorder",
    "Elliptocytosis 2": "Red Cell disorder",
    "Familial Hemolytic Anemia": "Red Cell disorder",
    "Heinz Body Anemia": "Red Cell disorder",
    "Heme Oxygenase 1 Deficiency": "Red Cell disorder",
    "Hemoglobin H Disease": "Red Cell disorder",
    "Hemoglobinopathy": "Red Cell disorder",
    "Hemolytic Anemia": "Red Cell disorder",
    "Hereditary Spherocytosis": "Red Cell disorder",
    "Macrocytic Anemia": "Red Cell disorder",
    "Megaloblastic Anemia": "Red Cell disorder",
    "Neonatal Anemia": "Red Cell disorder",
    "Overhydrated Hereditary Stomatocytosis": "Red Cell disorder",
    "Pernicious Anemia": "Red Cell disorder",
    "Polycythemia": "Red Cell disorder",
    "Pyropoikilocytosis": "Red Cell disorder",
    "Rh Deficiency Syndrome": "Red Cell disorder",
    "Sickle Cell Disease": "Red Cell disorder",
    "Thalassemia": "Red Cell disorder",


    # --- PLATELET / COAGULATION ---
    "Abnormal Bleeding": "MK/Platelet disorder",
    "Atypical Hemolytic Uremic Syndrome": "Immunodeficiency",   # CORRECTED (complement-mediated)
    "Bernard - Soulier Syndrome": "MK/Platelet disorder",
    "Beta-Thalassemia X-Linked Thrombocytopenia Syndrome": "Mixed or Haematopoietic disorder",
    "Blood Coagulation Disorder": "MK/Platelet disorder",
    "Coagulation Phenotype": "MK/Platelet disorder",
    "Factor XIII Subunit a Deficiency": "MK/Platelet disorder",
    "Fetal and Neonatal Alloimmune Thrombocytopenia": "MK/Platelet disorder",
    "Glanzmann Thrombasthenia": "MK/Platelet disorder",
    "Hemorrhagic Disease": "MK/Platelet disorder",
    "Hereditary Antithrombin Deficiency": "MK/Platelet disorder",
    "Impaired ADP-Induced Platelet Aggregation": "MK/Platelet disorder",
    "Platelet Disorder": "MK/Platelet disorder",
    "Platelet Storage Pool Disease": "MK/Platelet disorder",
    "Platelet-Type Bleeding Disorder": "MK/Platelet disorder",
    "Prolonged Bleeding Time": "MK/Platelet disorder",
    "Quebec Platelet Disorder": "MK/Platelet disorder",
    "Radio - Ulnar Synostosis - Amegakaryocytic Thrombocytopenia": "MK/Platelet disorder",
    "Reduced Antithrombin III Activity": "MK/Platelet disorder",
    "Thrombocytopenia": "MK/Platelet disorder",
    "Thrombophilia": "MK/Platelet disorder",
    "Thromboxane Synthesis Defect": "MK/Platelet disorder",
    "Von Willebrand Disease": "MK/Platelet disorder",


    # --- MYELOID ---
    "Abnormal Leukocyte Morphology": "Myeloid disorder",
    "Absence of Circulating Granulocytes": "Myeloid disorder",
    "Cyclic Neutropenia": "Myeloid disorder",
    "Hereditary Neutrophilia": "Myeloid disorder",
    "Juvenile Myelomonocytic Leukemia": "Myeloid disorder",
    "Monocytic Leukemia": "Myeloid disorder",
    "Monocytopenia with Susceptibility to Infections": "Myeloid disorder",
    "Monosomy 7 Myelodysplasia and Leukemia Syndrome": "Myeloid disorder",
    "Myelodysplastic Syndrome": "Myeloid disorder",
    "Myeloid Leukemia": "Myeloid disorder",
    "Myeloperoxidase Deficiency": "Myeloid disorder",
    "Myeloproliferative Disorder": "Myeloid disorder",
    "Neutropenia": "Myeloid disorder",
    "Sea - Blue Histiocytosis": "Myeloid disorder",
    "Severe Congenital Neutropenia": "Myeloid disorder",


    # --- LYMPHOID ---
    "Acute Lymphoblastic Leukemia": "Lymphoid disorder",
    "B-Cell Acute Lymphoblastic Leukemia": "Lymphoid disorder",
    "Chronic Lymphocytic Leukemia": "Lymphoid disorder",
    "Cutaneous Follicular Lymphoma": "Lymphoid disorder",
    "EBV-Positive Nodal T- and NK-Cell Lymphoma": "Lymphoid disorder",
    "Follicular Lymphoma": "Lymphoid disorder",
    "Hodgkin Lymphoma": "Lymphoid disorder",
    "Lymphoid Leukemia": "Lymphoid disorder",
    "Lymphoid Neoplasm": "Lymphoid disorder",
    "Lymphoma": "Lymphoid disorder",
    "Lymphopenia": "Lymphoid disorder",
    "Lymphoproliferative Syndrome": "Lymphoid disorder",
    "Monoclonal Gammopathy": "Lymphoid disorder",
    "Multiple Myeloma": "Lymphoid disorder",
    "Non-Hodgkin Lymphoma": "Lymphoid disorder",
    "Paraproteinemia": "Lymphoid disorder",
    "Subcutaneous Panniculitis-Like T-Cell Lymphoma": "Lymphoid disorder",
    "T-Cell Acute Lymphoblastic Leukemia": "Lymphoid disorder",


    # --- MIXED OR HAEMATOPOIETIC ---
    "Abnormality of Blood and Blood - Forming Tissues": "Mixed or Haematopoietic disorder",
    "Aplastic Anemia": "Mixed or Haematopoietic disorder",
    "Ataxia - Pancytopenia Syndrome": "Mixed or Haematopoietic disorder",  # CORRECTED
    "Bone Marrow Failure Syndrome": "Mixed or Haematopoietic disorder",
    "Bone Marrow Neoplasm": "Mixed or Haematopoietic disorder",
    "Clonal Hematopoiesis": "Mixed or Haematopoietic disorder",
    "Common Hematopoietic Neoplasm": "Mixed or Haematopoietic disorder",
    "Congenital Progressive Bone Marrow Failure - B - Cell Immunodeficiency - Skeletal Dysplasia Syndrome":
        "Mixed or Haematopoietic disorder",
    "Fanconi Anemia": "Mixed or Haematopoietic disorder",
    "GATA1-Related Cytopenia": "Mixed or Haematopoietic disorder",
    "GATA2 Deficiency with Susceptibility to MDS/AML": "Mixed or Haematopoietic disorder",  # CORRECTED
    "Hematologic Disease": "Mixed or Haematopoietic disorder",
    "Hematopoietic and Lymphoid Neoplasm": "Mixed or Haematopoietic disorder",
    "Leukemia": "Mixed or Haematopoietic disorder",
    "Pancytopenia due to IKZF1 Mutations": "Mixed or Haematopoietic disorder",
    "Shwachman-Diamond Syndrome": "Mixed or Haematopoietic disorder",
    "Thrombocytopenia, Anemia, and Myelofibrosis": "Mixed or Haematopoietic disorder",
    "X - Linked Dyserythropoetic Anemia with Abnormal Platelets and Neutropenia":
        "Mixed or Haematopoietic disorder",


    # --- IMMUNODEFICIENCIES ---
    "Activated PI3K - Delta Syndrome": "Immunodeficiency",
    "Agammaglobulinemia": "Immunodeficiency",
    "Aicardi - Goutières Syndrome": "Immunodeficiency",
    "Autoimmune Lymphoproliferative Syndrome": "Immunodeficiency",
    "Autoinflammatory Pancytopenia Syndrome": "Immunodeficiency",  # CORRECTED
    "Autoinflammatory Syndrome": "Immunodeficiency",
    "B - Cell Immunodeficiency, Distal Limb Anomalies, and Urogenital Malformations":
        "Immunodeficiency",
    "BENTA Disease": "Immunodeficiency",
    "CINCA Syndrome": "Immunodeficiency",
    "Chediak - Higashi Syndrome": "Immunodeficiency",
    "Chronic Granulomatous Disease": "Immunodeficiency",
    "Chronic Mucocutaneous Candidiasis": "Immunodeficiency",
    "Common Variable Immunodeficiency": "Immunodeficiency",
    "Complement Factor H Deficiency": "Immunodeficiency",
    "Ectodermal Dysplasia and Immunodeficiency 2": "Immunodeficiency",
    "Familial Hemophagocytic Lymphohistiocytosis": "Immunodeficiency",
    "Familial Variable Immunodeficiency": "Immunodeficiency",
    "Gamma Chain Deficiency": "Immunodeficiency",
    "HLA Class I Deficiency": "Immunodeficiency",
    "HLA Class II Deficiency": "Immunodeficiency",
    "Hyper-IgE Syndrome": "Immunodeficiency",
    "Hyper-IgM Syndrome": "Immunodeficiency",
    "ICF Syndrome": "Immunodeficiency",
    "IFIH1 - Related type 1 Interferonopathy": "Immunodeficiency",
    "IRAK4 Deficiency": "Immunodeficiency",
    "Immunodeficiency, Unspecified": "Immunodeficiency",
    "Immunoglobulin A Deficiency": "Immunodeficiency",
    "Leukocyte Adhesion Deficiency": "Immunodeficiency",
    "MHC Class I Deficiency": "Immunodeficiency",
    "MHC Class II Deficiency": "Immunodeficiency",
    "Monogenic Immunodeficiency": "Immunodeficiency",
    "Muckle - Wells Syndrome": "Immunodeficiency",
    "Mucocutaneous Lymph Node Syndrome": "Immunodeficiency",
    "NK Cell Deficiency with Adrenal Insufficiency": "Immunodeficiency",
    "Omenn Syndrome": "Immunodeficiency",
    "PLCG2-Associated Antibody Deficiency and Immune Dysregulation":
        "Immunodeficiency",
    "Properdin Deficiency": "Immunodeficiency",
    "Pulmonary Alveolar Proteinosis with Hypogammaglobulinemia":
        "Immunodeficiency",
    "Purine Nucleoside Phosphorylase Deficiency": "Immunodeficiency",
    "RAG2 Deficiency": "Immunodeficiency",
    "Recurrent Infections Associated with Rare Immunoglobulin Isotypes Deficiency":
        "Immunodeficiency",
    "Recurrent Neisseria Infections due to Factor D Deficiency":
        "Immunodeficiency",
    "Severe Combined Immunodeficiency": "Immunodeficiency",
    "Susceptibility to Viral and Mycobacterial Infections": "Immunodeficiency",
    "T - Cell Immunodeficiency with Epidermodysplasia Verruciformis":
        "Immunodeficiency",
    "WHIM Syndrome": "Immunodeficiency",
    "X - Linked Lymphoproliferative Disease": "Immunodeficiency",


    # --- SYSTEMIC / DEVELOPMENTAL ---
    "CHARGE Syndrome": "Systemic/Developmental",
    "Deafness-Lymphedema-Leukemia Syndrome": "Systemic/Developmental",
    "Dorfman - Chanarin Disease": "Systemic/Developmental",
    "Hennekam Lymphangiectasia - Lymphedema Syndrome 2": "Systemic/Developmental",
    "Jacobsen Syndrome": "Systemic/Developmental",
    "LIG4 Syndrome": "Systemic/Developmental",
    "Li-Fraumeni Syndrome": "Systemic/Developmental",
    "Lymphangioma": "Systemic/Developmental",
    "MIRAGE Syndrome": "Systemic/Developmental",
    "Polyglucosan Body Myopathy 1 with or without Immunodeficiency":
        "Systemic/Developmental",
    "Pseudo - TORCH Syndrome 3": "Systemic/Developmental",
    "Trypanosomiasis": "Systemic/Developmental",
}

hematological_df['lineage_category'] = hematological_df['disease_name_clean'].map(group_map)

/tmp/ipykernel_2749781/1424246192.py:197: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hematological_df['lineage_category'] = hematological_df['disease_name_clean'].map(group_map)


In [24]:
hematological_df.lineage_category.value_counts()

lineage_category
Lymphoid disorder                   3004
Red Cell disorder                    696
Immunodeficiency                     682
Myeloid disorder                     643
Mixed or Haematopoietic disorder     471
MK/Platelet disorder                 378
Systemic/Developmental                50
Name: count, dtype: int64

In [25]:
# concatenate measurement_df with hematological_df
combined_hematological_df = pd.concat([measurement_df, hematological_df], ignore_index=True)

In [26]:
combined_hematological_df = combined_hematological_df[combined_hematological_df['lineage_category'].notnull()]

In [27]:
combined_hematological_df

,Unnamed: 0,ensembl_id,genes,cluster,disease_id,disease_name,therapeutic_area_name,therapeutic_area_id,association_score,disease_ancestors,chembl,eva,gene_burden,genomics_england,gwas_credible_sets,lineage_category,disease_name_clean
0,19,ENSG00000000971,CFH,Fetal megakaryocyte - erythroid maturation,EFO_0004587,lymphocyte count,measurement,EFO_0001444,0.130458,"EFO_0004747, EFO_0004586, PATO_0000070, EFO_00...",NaN,NaN,NaN,NaN,0.429186,lymphoid,NaN
1,33,ENSG00000000971,CFH,Fetal megakaryocyte - erythroid maturation,EFO_0004833,neutrophil count,measurement,EFO_0001444,0.108375,"EFO_0004586, EFO_0803548, EFO_0004503, EFO_000...",NaN,NaN,NaN,NaN,0.356538,myeloid (granulocytic),NaN
2,35,ENSG00000000971,CFH,Fetal megakaryocyte - erythroid maturation,EFO_0005091,monocyte count,measurement,EFO_0001444,0.159970,"EFO_0803547, EFO_0004586, EFO_0001444, EFO_000...",NaN,NaN,NaN,NaN,0.526277,myeloid (monocytic),NaN
3,1045,ENSG00000001167,NFYA,Core transcription regulation,EFO_0004309,platelet count,measurement,EFO_0001444,0.009380,"EFO_0001444, EFO_0004503, EFO_0004586",NaN,NaN,NaN,NaN,0.030859,megakaryocytic,NaN
4,1046,ENSG00000001167,NFYA,Core transcription regulation,EFO_0004528,mean corpuscular hemoglobin concentration,measurement,EFO_0001444,0.104076,"EFO_0001444, EFO_0004306, EFO_0004509, EFO_000...",NaN,NaN,NaN,NaN,0.342393,erythroid,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22862,184847,ENSG00000259207,ITGB3,Early megakaryocyte - erythroid differentiation,MONDO_0100326,Glanzmann thrombasthenia,hematologic disease,EFO_0005803,0.628641,"MONDO_0000009, MONDO_0021181, EFO_0005803, OTA...",NaN,0.966133,NaN,0.759913,NaN,MK/Platelet disorder,Glanzmann Thrombasthenia
22863,184854,ENSG00000259207,ITGB3,Early megakaryocyte - erythroid differentiation,Orphanet_2314,Autosomal dominant hyper-IgE syndrome,"genetic, familial or congenital disease",OTAR_0000018,0.170007,"Orphanet_183494, MONDO_0021094, EFO_0000508, O...",NaN,0.559296,NaN,NaN,NaN,Immunodeficiency,Hyper-IgE Syndrome
22864,184855,ENSG00000259207,ITGB3,Early megakaryocyte - erythroid differentiation,Orphanet_2314,Autosomal dominant hyper-IgE syndrome,immune system disease,EFO_0000540,0.170007,"Orphanet_183494, MONDO_0021094, EFO_0000508, O...",NaN,0.559296,NaN,NaN,NaN,Immunodeficiency,Hyper-IgE Syndrome
22865,185557,ENSG00000276644,DACH1,Immune response to external stimuli,EFO_0004289,lymphoid leukemia,cancer or benign tumor,MONDO_0045024,0.027021,"MONDO_0045024, MONDO_0044881, EFO_0000565, EFO...",NaN,NaN,NaN,NaN,0.088896,Lymphoid disorder,Lymphoid Leukemia


In [28]:
combined_hematological_df = (
    combined_hematological_df
        .sort_values('association_score', ascending=False)  # highest first
        .drop_duplicates(subset=['genes', 'disease_name'])
)

In [90]:
# save to csv
combined_hematological_df.to_csv('hematological_disorders_annotated.csv', index=False)

In [29]:
combined_hematological_df.lineage_category.value_counts()

lineage_category
erythroid                           6794
megakaryocytic                      2879
lymphoid                            1889
myeloid (monocytic)                 1432
basophilic/eosinophilic             1384
myeloid (granulocytic)              1120
Lymphoid disorder                    909
myeloid                              507
Red Cell disorder                    370
MK/Platelet disorder                 305
Immunodeficiency                     282
Mixed or Haematopoietic disorder     237
Myeloid disorder                     226
lymphoid + myeloid (mixed index)      82
pan-hematologic                       26
Systemic/Developmental                20
pan-leukocytic                        15
Name: count, dtype: int64